In [1]:
import httpx
import time
import pandas as pd
import re
from transformers import pipeline
from datetime import datetime, timezone
import json
from pathlib import Path

/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.9.1 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0609 16:15:32.479000 58692 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
file_path = Path("/Users/mnatali/Projects/sentiment_analysis/brightdata_across_social_media_analysis/brightdata_social_exports/quora_datacenters_posts.json")

with file_path.open("r", encoding="utf-8") as f:
    quora_posts = json.load(f)

In [3]:
all_post_ids = []

env_post_ids = []
infr_post_ids = []
housing_post_ids = []
econ_post_ids = []
life_qual_post_ids = []
aesth_post_ids = []
gov_post_ids = []
not_useful_post_ids = []

environmental_keywords = ["environment", "pollution", "emissions", "diesel", "fumes", "light pollution", "water", "drought", "water consumption",  "waste", "carbon", "climate"]
infrastructure_utilities_keywords = ["utilities", "utility", "electric", "power", "grid", "blackout", "substation", "reliable", "reliability", "capacity", "infrastructure", "generator", "water"]
housing_keywords = ["housing", "house", "home", "rent", "property", "real estate", "house price", "housing price", "housing cost", "land cost", "land price", "gentrification"]
economic_keywords = ["job", "work", "revenue", "tax", "money", "rich", "salary", "economy", "investment", "business"]
life_quality_keywords = ["resident", "fumes", "noise", "loud", "quiet", "vibration", "light pollution", "sound", "smell", "traffic"]
aesthetics_keywords = ["landscape", "visual impact",  "aesthetics", "looks", "architecture", "design", "facade", "industrial", "windows", "dark"]
government_keywords = ["zoning", "zone", "planning", "approval", "permit", "city council", "bill", "mayor", "governor", "government"]

matched_posts = 0

for q_post in quora_posts:
    post_id = q_post["post_id"]
    all_post_ids.append(post_id)
    text = (q_post["post_text"])

    matched = False

    if any(kw in text for kw in environmental_keywords):
        env_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in infrastructure_utilities_keywords):
        infr_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in housing_keywords):
        housing_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in economic_keywords):
        econ_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in life_quality_keywords):
        life_qual_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in aesthetics_keywords):
        aesth_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in government_keywords):
        gov_post_ids.append(post_id)
        matched = True
    if not matched:
        not_useful_post_ids.append(post_id)
    else:
        matched_posts += 1

print("Total posts:", len(all_post_ids))
print("Total posts with a theme:", matched_posts)
print("Found environmental posts:", len(env_post_ids))
print("Found infrastructure posts:", len(infr_post_ids))
print("Found housing posts:", len(housing_post_ids))
print("Found economic posts:", len(econ_post_ids))
print("Found life quality posts:", len(life_qual_post_ids))
print("Found aesthetic posts:", len(aesth_post_ids))
print("Found government posts:", len(gov_post_ids))

posts_by_id = {post["post_id"]: post for post in quora_posts}

Total posts: 1300
Total posts with a theme: 1182
Found environmental posts: 284
Found infrastructure posts: 628
Found housing posts: 621
Found economic posts: 907
Found life quality posts: 478
Found aesthetic posts: 350
Found government posts: 325


In [4]:
theme_lists = [env_post_ids, infr_post_ids, housing_post_ids, econ_post_ids, life_qual_post_ids, aesth_post_ids, gov_post_ids]
posts = pd.DataFrame(columns=["ids", "text", "date", "upvotes", "number of answers", "shares", "views", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"])
datacenters_keywords = ["datacenter", "data center", "datacentre", "data centre"]

for theme in theme_lists:
    df1 = pd.DataFrame(columns=["ids", "text", "date", "upvotes", "number of answers", "shares", "views", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"])
    post_ids = []
    post_texts = []
    post_dates = []
    post_upvotes = []
    post_num_answers = []
    post_shares = []
    post_views = []


    for pid in theme:
        post_ids.append(pid)
        post = posts_by_id.get(pid)
        post_texts.append(post["post_text"])
        post_dates.append(post["post_date"])
        post_upvotes.append(post["upvotes"])
        post_num_answers.append(post["over_all_answers"])
        post_shares.append(post["shares"])
        post_views.append(post["views"])


    df1["ids"] = post_ids
    df1["text"] = post_texts
    df1["date"] = post_dates
    df1["upvotes"] = post_upvotes
    df1["number of answers"] = post_num_answers
    df1["shares"] = post_shares
    df1["views"] = post_views

    for col in ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]:
        df1[col] = False

    if theme == env_post_ids:
        df1["environment"] = True
    if theme == infr_post_ids:
        df1["infrastructure"] = True
    if theme == housing_post_ids:
        df1["housing"] = True
    if theme == econ_post_ids:
        df1["economy"] = True
    if theme == life_qual_post_ids:
        df1["life quality"] = True
    if theme == aesth_post_ids:
        df1["aesthetics"] = True
    if theme == gov_post_ids:
        df1["government"] = True
    posts = pd.concat([posts, df1], ignore_index=True)

len(posts['ids'])

3593

In [5]:
def remove_links(text):
    # Regex pattern to find typical URLs (http/https followed by non-whitespace characters)
    url_pattern = re.compile(r'\[https?://\S+\]|\(https?://\S+\)|\[www\.\S+\]|\(www\.\S+\)')
    # Replace found URLs with an empty string
    cleaned_text = url_pattern.sub('', text)
    return cleaned_text

sample_string = "The board’s land use policy committee heard a presentation from the Department of Planning and Development about potential enhancements to data center use standards, including requiring a noise study and establishing a minimum distance from residential areas. The board had directed staff last May to provide research, findings and recommendations...  McKay said this process is designed to steer data centers into places that are most appropriate and make sure standards are set for environmental protections, noise mitigation and other issues that have people concerned. He said the county is learning from Loudoun and Prince William Counties, where data centers are becoming more and more prevalent...  [https://wjla.com/news/local/fairfax-county-considers-enhancing-data-center-guidelines-virginia-community-development-board-of-supervisors#](https://wjla.com/news/local/fairfax-county-considers-enhancing-data-center-guidelines-virginia-community-development-board-of-supervisors#)  A group of residents from the [Save Bren Mar From Data Center](https://sites.google.com/view/savebrenmar/home) group from the Mason District held up signs throughout Tuesday's legislative policy committee meeting, calling calling on the Fairfax county Supervisors to end by right data center development on commercial and industrial properties.  [https://patch.com/virginia/reston/data-center-zoning-policy-be-updated-fairfax-county-board](https://patch.com/virginia/reston/data-center-zoning-policy-be-updated-fairfax-county-board)Fairfax County considers enhancing data center guidelines"
result = remove_links(sample_string)
print(result)

posts["text"] = posts["text"].apply(remove_links)

The board’s land use policy committee heard a presentation from the Department of Planning and Development about potential enhancements to data center use standards, including requiring a noise study and establishing a minimum distance from residential areas. The board had directed staff last May to provide research, findings and recommendations...  McKay said this process is designed to steer data centers into places that are most appropriate and make sure standards are set for environmental protections, noise mitigation and other issues that have people concerned. He said the county is learning from Loudoun and Prince William Counties, where data centers are becoming more and more prevalent...    A group of residents from the [Save Bren Mar From Data Center] group from the Mason District held up signs throughout Tuesday's legislative policy committee meeting, calling calling on the Fairfax county Supervisors to end by right data center development on commercial and industrial propert

In [6]:
grouping_cols = ["ids", "text", "date"]
posts = posts.groupby(grouping_cols, as_index=False).max()
theme_cols = ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]
len(posts['ids'])

1182

In [7]:
posts.head()

,ids,text,date,upvotes,number of answers,shares,views,environment,infrastructure,housing,economy,life quality,aesthetics,government
0,QW5zd2VyQDA6MTA0ODM2NDY1,Software-defined network (SDN) and a tradition...,2018-10-21T11:20:20.865Z,0,6,0,1317,False,False,True,True,True,True,False
1,QW5zd2VyQDA6MTA0OTM5NTk1,I don’t know who is the most clever person to ...,2018-10-22T05:07:17.756Z,518,51,14,58942,False,False,True,True,False,False,False
2,QW5zd2VyQDA6MTA1MDgyOTE4,"The original idea social networks, connecting ...",2018-10-23T06:07:33.317Z,1,6,0,1100,True,False,False,True,True,False,False
3,QW5zd2VyQDA6MTA1ODI1Mzg4,Pre-Sales and Bid-Management have immense pote...,2018-10-28T18:50:07.259Z,87,3,2,35079,True,True,True,True,True,True,False
4,QW5zd2VyQDA6MTA2MjEzNzA4,"In general, they made good updates to all but ...",2018-10-31T14:26:19.515Z,1,2,0,250,False,False,False,True,False,True,False


In [35]:
# def convert_dates(utc_date):
#     date = datetime.fromtimestamp(utc_date, tz=timezone.utc)
#     return date

# posts['date'] = posts['date'].apply(convert_dates)
# posts.head(10)

In [8]:
classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,      # <--- important
    padding=True,         # <--- for batching
    max_length=512        
)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [9]:
texts = posts["text"].astype(str).tolist()
results = classifier(texts, truncation=True, padding=True, max_length=512, batch_size=32)
posts["roberta"] = results

For reference, doing sentiment analysis on 1182 posts took 1 minutes and 45 seconds

In [10]:
posts["name"] = posts["roberta"].apply(lambda x: x["label"])

def roberta_label(name):
    if name == 'LABEL_2':
        return "positive"
    elif name == 'LABEL_1':
        return "neutral"
    else:
        return "negative"
posts["label"] = posts["name"].apply(roberta_label)

posts["degree"] = posts["roberta"].apply(lambda x: x["score"])
posts = posts.drop(["roberta", "name"], axis=1)
posts.head(15)

,ids,text,date,upvotes,number of answers,shares,views,environment,infrastructure,housing,economy,life quality,aesthetics,government,label,degree
0,QW5zd2VyQDA6MTA0ODM2NDY1,Software-defined network (SDN) and a tradition...,2018-10-21T11:20:20.865Z,0,6,0,1317,False,False,True,True,True,True,False,negative,0.702790
1,QW5zd2VyQDA6MTA0OTM5NTk1,I don’t know who is the most clever person to ...,2018-10-22T05:07:17.756Z,518,51,14,58942,False,False,True,True,False,False,False,negative,0.666412
2,QW5zd2VyQDA6MTA1MDgyOTE4,"The original idea social networks, connecting ...",2018-10-23T06:07:33.317Z,1,6,0,1100,True,False,False,True,True,False,False,negative,0.692488
3,QW5zd2VyQDA6MTA1ODI1Mzg4,Pre-Sales and Bid-Management have immense pote...,2018-10-28T18:50:07.259Z,87,3,2,35079,True,True,True,True,True,True,False,negative,0.533782
4,QW5zd2VyQDA6MTA2MjEzNzA4,"In general, they made good updates to all but ...",2018-10-31T14:26:19.515Z,1,2,0,250,False,False,False,True,False,True,False,negative,0.867723
5,QW5zd2VyQDA6MTA2MzYxOTMw,"That is a big subject, depending on the scale ...",2018-11-01T16:59:46.862Z,0,2,1,250,True,False,False,True,False,False,False,negative,0.773825
6,QW5zd2VyQDA6MTA2MzYyMjMw,"Since about 2016, maybe 2015, I have been seei...",2018-11-01T17:02:28.168Z,0,3,0,209,False,True,False,True,False,False,False,negative,0.743817
7,QW5zd2VyQDA6MTA2NzE5MzI=,"While it is possible to have similar climates,...",2015-03-20T17:00:49.216Z,2,2,0,370,True,False,False,False,False,False,False,negative,0.843803
8,QW5zd2VyQDA6MTA3MzIyNzIz,"I’ll give you two choices (or options, if you ...",2018-11-09T03:12:22.372Z,0,8,0,6798,False,False,True,True,False,False,False,negative,0.845235
9,QW5zd2VyQDA6MTA3NjY0ODc1,"A2A. All will benefit in small ways, but the b...",2018-11-11T22:26:52.306Z,9,114,0,771,False,True,True,True,False,True,True,negative,0.586592


In [12]:
# calculating average sentiment based on theme:
# Weigh all posts by their degree in the numerator and denominator, means that the average sentiment will just be +/- 1 if there are only positive or negative themes but other than that does a pretty good job of weighing neutrality
def avg_sentiment_calculation(theme):
    theme_posts = posts[posts[theme] == True]
    pos = theme_posts.loc[theme_posts["label"] == "positive", "degree"].sum()
    neg = theme_posts.loc[theme_posts["label"] == "negative", "degree"].sum()
    total = theme_posts["degree"].sum()
    return (pos-neg)/total

print("Average sentiment of environmental posts:", avg_sentiment_calculation("environment"))
print("Average sentiment of infrastructural posts:", avg_sentiment_calculation("infrastructure"))
print("Average sentiment of housing-related posts:", avg_sentiment_calculation("housing"))
print("Average sentiment of economic posts:", avg_sentiment_calculation("economy"))
print("Average sentiment of life-quality-related posts:", avg_sentiment_calculation("life quality"))
print("Average sentiment of aesthetics-related posts:", avg_sentiment_calculation("aesthetics"))
print("Average sentiment of governmental posts:", avg_sentiment_calculation("government"))

Average sentiment of environmental posts: -1.0
Average sentiment of infrastructural posts: -1.0
Average sentiment of housing-related posts: -1.0
Average sentiment of economic posts: -1.0
Average sentiment of life-quality-related posts: -1.0
Average sentiment of aesthetics-related posts: -1.0
Average sentiment of governmental posts: -1.0


In [13]:
## total average sentiment calculation

pos = posts.loc[posts["label"] == "positive", "degree"].sum()
neg = posts.loc[posts["label"] == "negative", "degree"].sum()
total = posts["degree"].sum()
print("Average sentiment for the whole subreddit: ", (pos-neg)/total)

Average sentiment for the whole subreddit:  -1.0


In [14]:
pos = posts.loc[posts["label"] == "positive"]
neg = posts.loc[posts["label"] == "negative"]
neutral = posts.loc[posts["label"] == "neutral"]

print("Percent of posts that are neutral:", len(neutral)/len(posts))
print("Percent of posts that are negative:", len(neg)/len(posts))
print("Percent of posts that are positive:", len(pos)/len(posts))

really_pos = pos.loc[posts["degree"] > 0.70]
really_neg = neg.loc[posts["degree"] > 0.70]

if len(neg) != 0:
    print("Percent of negative posts that are really negative:", len(really_neg)/len(neg))
if len(pos) != 0:
    print("Percent of positive posts that are really positive:", len(really_pos)/len(pos))

Percent of posts that are neutral: 0.0
Percent of posts that are negative: 1.0
Percent of posts that are positive: 0.0
Percent of negative posts that are really negative: 0.5152284263959391


In [15]:
def convert_dates(utc_date):
    date = datetime.fromtimestamp(utc_date, tz=timezone.utc)
    return date

In [16]:
pre_2010 = posts.loc[posts["date"] < 1262304000]

if len(pre_2010) == 0:
    print("None")
else:
    pre_2010['date'] = pre_2010['date'].apply(convert_dates)
    pre_2010.head(10)
    pos = pre_2010.loc[posts["label"] == "positive", "degree"].sum()
    neg = pre_2010.loc[posts["label"] == "negative", "degree"].sum()
    total = pre_2010["degree"].sum()
    print("Average sentiment for the whole subreddit: ", (pos-neg)/total)

TypeError: '<' not supported between instances of 'str' and 'int'

In [46]:
between_2010_2015 = posts.loc[(posts["date"] > 1262304000) & (posts["date"] < 1420070400)]

if len(between_2010_2015) == 0:
    print("None")
else:
    between_2010_2015['date'] = between_2010_2015['date'].apply(convert_dates)
    between_2010_2015.head(10)
    pos = between_2010_2015.loc[posts["label"] == "positive", "degree"].sum()
    neg = between_2010_2015.loc[posts["label"] == "negative", "degree"].sum()
    total = between_2010_2015["degree"].sum()
    print("Average sentiment for the whole subreddit: ", (pos-neg)/total)

None


In [47]:
between_2015_2020 = posts.loc[(posts["date"] > 1420070400) & (posts["date"] < 1577836800)]

if len(between_2015_2020) == 0:
    print("None")
else:
    between_2015_2020['date'] = between_2015_2020['date'].apply(convert_dates)
    between_2015_2020.head(10)
    pos = between_2015_2020.loc[posts["label"] == "positive", "degree"].sum()
    neg = between_2015_2020.loc[posts["label"] == "negative", "degree"].sum()
    total = between_2015_2020["degree"].sum()
    print("Average sentiment for the whole subreddit: ", (pos-neg)/total)

Average sentiment for the whole subreddit:  -1.0


/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_89670/2183586022.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  between_2015_2020['date'] = between_2015_2020['date'].apply(convert_dates)


In [48]:
between_2020_2025 = posts.loc[(posts["date"] > 1577836800) & (posts["date"] < 1735689600)]

if len(between_2020_2025) == 0:
    print("None")
else:
    between_2020_2025['date'] = between_2020_2025['date'].apply(convert_dates)
    between_2020_2025.head(10)
    pos = between_2020_2025.loc[posts["label"] == "positive", "degree"].sum()
    neg = between_2020_2025.loc[posts["label"] == "negative", "degree"].sum()
    total = between_2020_2025["degree"].sum()
    print("Average sentiment for the whole subreddit: ", (pos-neg)/total)

Average sentiment for the whole subreddit:  -1.0


/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_89670/3710434052.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  between_2020_2025['date'] = between_2020_2025['date'].apply(convert_dates)


In [49]:
between_2025_2026 = posts.loc[posts["date"] > 1735689600]

if len(between_2025_2026) == 0:
    print("None")
else:
    between_2025_2026['date'] = between_2025_2026['date'].apply(convert_dates)
    between_2025_2026.head(10)
    pos = between_2025_2026.loc[posts["label"] == "positive", "degree"].sum()
    neg = between_2025_2026.loc[posts["label"] == "negative", "degree"].sum()
    total = between_2025_2026["degree"].sum()
    print("Average sentiment for the whole subreddit: ", (pos-neg)/total)

Average sentiment for the whole subreddit:  -1.0


/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_89670/1067228198.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  between_2025_2026['date'] = between_2025_2026['date'].apply(convert_dates)
